# prepare a full rho/temp data cube

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.constants import k as k_B_SI

from cooling_functions import cool_Lambda, cooling_time, volumetric_cooling_rate, \
make_equilibrium_temperature_function, make_heating_from_equilibrium_temperature

k_B = k_B_SI * 1e7        # erg / K
Gamma = 2e-26             # erg / s, heating rate per H nucleus

from astropy import units as u, constants  as c

pc = c.pc.cgs.value
kB  = c.k_B.cgs.value
Msun = c.M_sun.cgs.value
G = c.G.cgs.value
Myr = u.Myr.to("s")
mp = c.m_p.cgs.value

# ------------------------------------------------------------
# 1. Read artificial density field
# ------------------------------------------------------------

rho_raw = np.load("data/density.npy")     # example: shape (Nx, Ny, Nz)
print("min/mean/median/max density in raw data:", np.min(rho_raw),np.mean(rho_raw),np.median(rho_raw),np.max(rho_raw))

min/mean/median/max density in raw data: 0.009170629159553762 0.9905001513021596 0.6214547096551986 37.72541062873586


In [2]:
# ------------------------------------------------------------
# 2. Scale to desired mean number density
# ------------------------------------------------------------

n_mean_target = 1.0   # cm^-3

rho_positive = np.maximum(rho_raw, 1e-30)

n_field   = rho_positive / np.mean(rho_positive) * n_mean_target
rho_field = n_field * 1.4 * mp
print("density shape:", n_field.shape)
print("min / mean / max n:",
      np.min(n_field), np.mean(n_field), np.max(n_field))
print("min / mean / max rho:",
      np.min(rho_field), np.mean(rho_field), np.max(rho_field))

density shape: (128, 128, 128)
min / mean / max n: 0.009258584309651652 0.9999999999999999 38.087233585113744
min / mean / max rho: 2.1680555538117832e-26 2.3416706931659996e-24 8.918775867002865e-23


In [3]:
#load eq. temp from SILCC data
import pickle

with open('../sim-data/SILCC-equilibrium-temperature.pkl', 'rb') as handle:
    data_Teq = pickle.load(handle)
print(data_Teq)

{'logTeq': array([6.14070352, 6.0201005 , 5.98994975, 5.71859296, 4.51256281,
       4.03015075, 3.78894472, 3.69849246, 3.6080402 , 2.88442211,
       2.46231156, 2.19095477, 1.9798995 , 1.79899497, 1.64824121,
       1.52763819, 1.43718593, 1.37688442, 1.40703518]), 'logrho': array([-26.81578947, -26.44736842, -26.07894737, -25.71052632,
       -25.34210526, -24.97368421, -24.60526316, -24.23684211,
       -23.86842105, -23.5       , -23.13157895, -22.76315789,
       -22.39473684, -22.02631579, -21.65789474, -21.28947368,
       -20.92105263, -20.55263158, -20.18421053]), 'logn': array([6.52656557e-04, 1.52442393e-03, 3.56062971e-03, 8.31663925e-03,
       1.94253528e-02, 4.53722137e-02, 1.05976854e-01, 2.47532412e-01,
       5.78166770e-01, 1.35043654e+00, 3.15424361e+00, 7.36743452e+00,
       1.72082750e+01, 4.01937373e+01, 9.38813752e+01, 2.19280744e+02,
       5.12178740e+02, 1.19630688e+03, 2.79423966e+03])}


In [4]:
logT_field = np.interp(np.log10(n_field), data_Teq["logn"], data_Teq["logTeq"])

In [5]:
print("min / mean / max logT:",
      np.min(logT_field), np.mean(logT_field), np.max(logT_field))

min / mean / max logT: 2.830519358257223 5.420059568241046 6.140703517587939


In [6]:
T_field = np.power(10.0, logT_field)

In [7]:
print("min / mean / max logT:",
      np.min(T_field), np.mean(T_field), np.max(T_field))

min / mean / max logT: 676.8919640530803 959964.6137342046 1382622.1737646535


In [8]:
T_field.shape

(128, 128, 128)

# save the data product

In [9]:
data_cooling = {}
data_cooling["n"]   = n_field
data_cooling["rho"] = rho_field
data_cooling["T"]   = T_field

with open('../tmp-data/scaled-density-Teq-cube.pkl', 'wb') as handle:
    pickle.dump(data_cooling, handle, protocol=pickle.HIGHEST_PROTOCOL)
